# 40 — Self-Healing CI Agent

A bounded agentic repair loop that reads CI failure logs, classifies the error type, selects a repair strategy, applies it, and re-validates — looping up to N retries before emitting a structured postmortem on terminal failure. The entire loop is explicit Python using the OpenAI SDK directly, with no external orchestration framework.

## Framework comparison — bounded repair loop pattern

| Framework | Equivalent construct |
|-----------|----------------------|
| LangGraph | Conditional edges + state machine with a retry counter in graph state |
| LangChain | Agent executor with memory and a custom retry tool |
| CrewAI | Sequential tasks with retry logic wired between crew steps |

In [ ]:
!pip install openai pydantic python-dotenv

In [ ]:
import os

# Replace the value below with your actual OpenAI API key before running
os.environ["OPENAI_API_KEY"] = "sk-..."

In [ ]:
"""
Pydantic models for the self-healing CI agent.

Covers the full repair-loop lifecycle:
  CIFailure -> FailureClassification -> RepairAttempt(s) -> RepairPostmortem -> HealingResult
"""
from __future__ import annotations

from typing import Literal

from pydantic import BaseModel, Field

ErrorType = Literal["dependency", "config", "code", "test", "flaky", "unknown"]
RepairStrategy = Literal[
    "pin_dependency", "update_config", "fix_syntax", "skip_test", "retry", "escalate"
]


class CIFailure(BaseModel):
    """Input: a single CI job failure to be healed."""

    log_snippet: str = Field(
        description="Relevant excerpt from the CI job log (stderr, traceback, exit output)."
    )
    job_name: str = Field(
        description="Name of the CI job that failed (e.g. 'build', 'test', 'lint')."
    )
    step_name: str = Field(
        description="Name of the step within the job that produced the failure."
    )
    exit_code: int = Field(
        description="Process exit code returned by the failing step."
    )


class FailureClassification(BaseModel):
    """LLM output: classification of a CI failure."""

    error_type: ErrorType = Field(
        description=(
            "Category of the failure: dependency (missing/conflicting package), "
            "config (misconfigured env or CI YAML), code (syntax or runtime bug), "
            "test (deterministic test assertion failure), flaky (non-deterministic / "
            "intermittent failure), or unknown."
        )
    )
    root_cause: str = Field(
        description="One-sentence description of the most likely root cause."
    )
    suggested_strategy: RepairStrategy = Field(
        description=(
            "Recommended first repair strategy: pin_dependency, update_config, "
            "fix_syntax, skip_test, retry, or escalate."
        )
    )
    confidence: float = Field(
        description=(
            "Confidence score for this classification, between 0.0 (no confidence) "
            "and 1.0 (certain)."
        ),
        ge=0.0,
        le=1.0,
    )


class RepairAttempt(BaseModel):
    """Record of a single repair iteration."""

    attempt_number: int = Field(
        description="1-based index of this attempt within the current healing run."
    )
    strategy: RepairStrategy = Field(
        description="Repair strategy applied in this attempt."
    )
    action_taken: str = Field(
        description=(
            "Concrete action performed (e.g. 'pinned requests==2.31.0 in requirements.txt')."
        )
    )
    patch_description: str = Field(
        description=(
            "Human-readable description of what the patch changes and why it should "
            "resolve the failure."
        )
    )
    validation_result: Literal["pass", "fail", "inconclusive"] = Field(
        description=(
            "Outcome after applying the patch: pass (issue resolved), "
            "fail (patch did not help), or inconclusive (cannot determine)."
        )
    )
    notes: str = Field(
        description=(
            "Additional context, warnings, or rationale from the LLM about this attempt."
        )
    )


class RepairPostmortem(BaseModel):
    """Terminal postmortem emitted when all retries are exhausted."""

    job_name: str = Field(
        description="Name of the CI job that could not be healed."
    )
    error_type: ErrorType = Field(
        description="Error category as determined during the initial classification step."
    )
    attempts: list[RepairAttempt] = Field(
        description="All repair attempts made before giving up, in order."
    )
    terminal_failure: bool = Field(
        description=(
            "Always True for a postmortem — indicates the agent could not resolve "
            "the issue."
        )
    )
    root_cause: str = Field(
        description="Final assessment of the root cause after all repair attempts."
    )
    recommended_fix: str = Field(
        description=(
            "Best recommended fix for a human engineer to apply manually."
        )
    )
    escalation_notes: str = Field(
        description=(
            "Context and next steps for the on-call engineer or team receiving "
            "this escalation."
        )
    )


class HealingResult(BaseModel):
    """Top-level result returned by the orchestrator's run() function."""

    job_name: str = Field(
        description="Name of the CI job that was processed."
    )
    resolved: bool = Field(
        description=(
            "True if the agent successfully healed the failure, False if retries "
            "were exhausted."
        )
    )
    attempts_taken: int = Field(
        description="Total number of repair attempts made."
    )
    final_strategy: RepairStrategy | None = Field(
        default=None,
        description="Strategy that resolved the failure, or None if unresolved.",
    )
    postmortem: RepairPostmortem | None = Field(
        default=None,
        description="Structured postmortem, populated only when resolved=False.",
    )

In [ ]:
"""
System prompt constants for the self-healing CI agent.

Each constant is consumed by exactly one function in the workflow:
  CLASSIFY_SYSTEM   -> _classify()
  REPAIR_SYSTEM     -> _propose_repair()
  VALIDATE_SYSTEM   -> _validate()
  POSTMORTEM_SYSTEM -> _write_postmortem()
"""

CLASSIFY_SYSTEM = """
You are a CI failure analyst. You receive a CI job log snippet and must classify
the failure so a repair agent can act on it.

Return a JSON object that conforms to the FailureClassification schema:
  - error_type: one of "dependency", "config", "code", "test", "flaky", "unknown"
  - root_cause: exactly one sentence identifying the most probable cause
  - suggested_strategy: one of "pin_dependency", "update_config", "fix_syntax",
    "skip_test", "retry", "escalate"
  - confidence: float 0.0-1.0 reflecting how certain you are

Strategy guide:
  dependency  -> pin_dependency (lock to a known-good version)
  config      -> update_config (fix env var, YAML key, or secret reference)
  code        -> fix_syntax (correct the bad import, typo, or logic error)
  test        -> skip_test (mark flaky/broken test as skipped pending a fix)
  flaky       -> retry (re-run the step; no code change needed)
  unknown     -> escalate (cannot determine cause; hand off to a human)

Respond with valid JSON only. Do not include prose outside the JSON object.
""".strip()

REPAIR_SYSTEM = """
You are a CI repair agent. You receive:
  1. A FailureClassification describing the error type, root cause, and suggested strategy.
  2. A JSON array of previous RepairAttempt objects (may be empty on the first attempt).

Your task is to propose the next concrete repair action.

Return a JSON object that conforms to the RepairAttempt schema:
  - attempt_number: integer, one more than the last attempt (1 if no prior attempts)
  - strategy: the repair strategy you are applying
  - action_taken: a precise, imperative description of the exact change to make
    (e.g. "Add requests==2.31.0 to requirements.txt and remove the unpinned entry")
  - patch_description: 1-2 sentences explaining what this patch changes and why it
    should fix the root cause
  - validation_result: set to "inconclusive" — the orchestrator will fill this in
  - notes: any caveats, risks, or follow-up observations

Rules:
  - Do not repeat a strategy that already appeared in the attempt history and
    returned "fail". Escalate if all reasonable strategies are exhausted.
  - Keep action_taken concrete and specific — not vague like "fix the dependency".
  - Respond with valid JSON only.
""".strip()

VALIDATE_SYSTEM = """
You are a CI patch validator. You receive:
  1. A RepairAttempt describing the action taken and the patch applied.
  2. The original CI log snippet that triggered the repair loop.

Assess whether the described patch would resolve the original failure if applied.

Return a JSON object with exactly two keys:
  - validation_result: "pass" if the patch is likely to fix the issue,
    "fail" if it will not, or "inconclusive" if you cannot determine
  - notes: 1-2 sentences explaining your verdict, including any remaining risk

Criteria for "pass":
  - The action_taken directly addresses the root cause identified in the log.
  - No obvious side effects or missing steps.
  - The strategy is appropriate for the error type.

Criteria for "fail":
  - The patch targets the wrong component or a symptom, not the cause.
  - A previous identical strategy already failed.
  - The patch description is contradicted by the log evidence.

Respond with valid JSON only. Do not include prose outside the JSON object.
""".strip()

POSTMORTEM_SYSTEM = """
You are a senior reliability engineer writing a terminal postmortem after a
self-healing CI agent exhausted all repair retries without resolving a failure.

You receive:
  1. The original CIFailure (job_name, step_name, exit_code, log_snippet).
  2. The FailureClassification from the first analysis step.
  3. A JSON array of all RepairAttempt objects, each with its validation_result.

Write a RepairPostmortem that conforms to the schema:
  - job_name: from the CIFailure input
  - error_type: from the FailureClassification
  - attempts: pass through the full list of RepairAttempt objects as provided
  - terminal_failure: always true
  - root_cause: your final, most informed assessment after reviewing all attempts
  - recommended_fix: the single most actionable fix a human engineer should apply,
    written as a concrete instruction (not "investigate further")
  - escalation_notes: 2-3 sentences of context for the on-call engineer describing
    what was tried, what failed, and what additional information they should gather

Tone: blameless, precise, and actionable. No speculation beyond what the evidence supports.
Respond with valid JSON only.
""".strip()

In [ ]:
"""
Self-healing CI agent — bounded ReAct repair loop.

Loop structure (max_retries iterations):
  1. classify()       -- identify error type, root cause, suggested strategy
  2. propose_repair() -- propose a concrete patch based on classification + history
  3. validate()       -- assess whether the patch resolves the original failure
  4. if pass: return HealingResult(resolved=True)
  5. if retries exhausted: write_postmortem() -> HealingResult(resolved=False)

All LLM calls use structured JSON output via OpenAI response_format.
"""
from __future__ import annotations

import json
import os

from openai import OpenAI

_MODEL = "gpt-4o-mini"


def _get_client() -> OpenAI:
    return OpenAI(api_key=os.environ["OPENAI_API_KEY"])


def _classify(failure: CIFailure) -> FailureClassification:
    """Call the LLM to classify the CI failure."""
    client = _get_client()
    user_content = (
        f"Job: {failure.job_name}\n"
        f"Step: {failure.step_name}\n"
        f"Exit code: {failure.exit_code}\n\n"
        f"Log snippet:\n{failure.log_snippet}"
    )
    resp = client.chat.completions.create(
        model=_MODEL,
        messages=[
            {"role": "system", "content": CLASSIFY_SYSTEM},
            {"role": "user", "content": user_content},
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "FailureClassification",
                "strict": True,
                "schema": FailureClassification.model_json_schema(),
            },
        },
    )
    return FailureClassification.model_validate_json(resp.choices[0].message.content)


def _propose_repair(
    failure: CIFailure,
    classification: FailureClassification,
    attempts: list[RepairAttempt],
) -> RepairAttempt:
    """Call the LLM to propose the next repair action."""
    client = _get_client()
    user_content = json.dumps(
        {
            "failure": failure.model_dump(),
            "classification": classification.model_dump(),
            "prior_attempts": [a.model_dump() for a in attempts],
        },
        indent=2,
    )
    resp = client.chat.completions.create(
        model=_MODEL,
        messages=[
            {"role": "system", "content": REPAIR_SYSTEM},
            {"role": "user", "content": user_content},
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "RepairAttempt",
                "strict": True,
                "schema": RepairAttempt.model_json_schema(),
            },
        },
    )
    attempt = RepairAttempt.model_validate_json(resp.choices[0].message.content)
    attempt.notes = attempt.notes or "pending"
    return attempt


def _validate(failure: CIFailure, attempt: RepairAttempt) -> RepairAttempt:
    """Validate whether the repair attempt resolves the failure.

    Fills attempt.validation_result and attempt.notes and returns the updated attempt.
    """
    client = _get_client()
    user_content = json.dumps(
        {
            "attempt": attempt.model_dump(),
            "original_log_snippet": failure.log_snippet,
        },
        indent=2,
    )
    resp = client.chat.completions.create(
        model=_MODEL,
        messages=[
            {"role": "system", "content": VALIDATE_SYSTEM},
            {"role": "user", "content": user_content},
        ],
        response_format={"type": "json_object"},
    )
    verdict = json.loads(resp.choices[0].message.content)
    attempt.validation_result = verdict.get("validation_result", "inconclusive")
    attempt.notes = verdict.get("notes", attempt.notes)
    return attempt


def _write_postmortem(
    failure: CIFailure,
    classification: FailureClassification,
    attempts: list[RepairAttempt],
) -> RepairPostmortem:
    """Write a terminal postmortem after retries are exhausted."""
    client = _get_client()
    user_content = json.dumps(
        {
            "failure": failure.model_dump(),
            "classification": classification.model_dump(),
            "attempts": [a.model_dump() for a in attempts],
        },
        indent=2,
    )
    resp = client.chat.completions.create(
        model=_MODEL,
        messages=[
            {"role": "system", "content": POSTMORTEM_SYSTEM},
            {"role": "user", "content": user_content},
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "RepairPostmortem",
                "strict": True,
                "schema": RepairPostmortem.model_json_schema(),
            },
        },
    )
    return RepairPostmortem.model_validate_json(resp.choices[0].message.content)


def run(failure: CIFailure, max_retries: int = 3) -> HealingResult:
    """
    Run the bounded self-healing loop for a CI failure.

    Steps:
      1. Classify the failure.
      2. Loop up to max_retries times:
         a. Propose a repair.
         b. Validate the repair.
         c. If validation_result == "pass", return resolved HealingResult.
      3. If the loop exhausts retries, write a postmortem and return unresolved.

    Args:
        failure:     The CI failure to heal.
        max_retries: Maximum repair attempts before giving up (default 3).

    Returns:
        HealingResult with resolved=True and final_strategy on success, or
        resolved=False with a populated postmortem on terminal failure.
    """
    classification = _classify(failure)
    attempts: list[RepairAttempt] = []

    for _ in range(max_retries):
        attempt = _propose_repair(failure, classification, attempts)
        attempt = _validate(failure, attempt)
        attempts.append(attempt)

        if attempt.validation_result == "pass":
            return HealingResult(
                job_name=failure.job_name,
                resolved=True,
                attempts_taken=len(attempts),
                final_strategy=attempt.strategy,
            )

    postmortem = _write_postmortem(failure, classification, attempts)
    return HealingResult(
        job_name=failure.job_name,
        resolved=False,
        attempts_taken=len(attempts),
        final_strategy=None,
        postmortem=postmortem,
    )

In [ ]:
# ---------------------------------------------------------------------------
# Scenario A — dependency conflict (expected to resolve)
# ---------------------------------------------------------------------------
SCENARIO_A = CIFailure(
    job_name="build",
    step_name="install-dependencies",
    exit_code=1,
    log_snippet=(
        "Collecting dependencies from requirements.txt\n"
        "ERROR: Could not find a version that satisfies the requirement requests>=2.28.0\n"
        "ERROR: No matching distribution found for requests>=2.28.0\n"
        "ModuleNotFoundError: No module named 'requests'\n"
        "Command 'pip install -r requirements.txt' returned non-zero exit status 1."
    ),
)


def _print_result(label: str, failure: CIFailure) -> None:
    print(f"\n{'=' * 60}")
    print(f"Scenario {label}: {failure.job_name} / {failure.step_name}")
    print("=" * 60)

    result = run(failure, max_retries=3)

    print(f"Job:           {result.job_name}")
    print(f"Resolved:      {result.resolved}")
    print(f"Attempts:      {result.attempts_taken}")

    if result.resolved:
        print(f"Final strategy: {result.final_strategy}")
    else:
        pm = result.postmortem
        if pm:
            print("\n--- Postmortem ---")
            print(f"Error type:      {pm.error_type}")
            print(f"Root cause:      {pm.root_cause}")
            print(f"Recommended fix: {pm.recommended_fix}")
            print(f"Escalation:      {pm.escalation_notes}")
            print(f"\nAttempts made: {len(pm.attempts)}")
            for attempt in pm.attempts:
                print(
                    f"  [{attempt.attempt_number}] {attempt.strategy} "
                    f"-> {attempt.validation_result}: {attempt.action_taken}"
                )


_print_result("A", SCENARIO_A)

In [ ]:
# ---------------------------------------------------------------------------
# Scenario B — flaky test (expected to exhaust retries and emit postmortem)
# ---------------------------------------------------------------------------
SCENARIO_B = CIFailure(
    job_name="test",
    step_name="run-unit-tests",
    exit_code=1,
    log_snippet=(
        "FAILED tests/test_payment_processor.py::test_stripe_charge_idempotency\n"
        "AssertionError: assert response.status_code == 200\n"
        "  where response.status_code = 500\n"
        "E   requests.exceptions.ConnectionError: "
        "HTTPSConnectionPool(host='api.stripe.com'): "
        "Max retries exceeded with url: /v1/charges\n"
        "FAILED tests/test_payment_processor.py::test_stripe_charge_idempotency - "
        "ConnectionError\n"
        "1 failed, 47 passed in 12.34s\n"
        "This test fails intermittently on CI runners due to network timeouts. "
        "Retry count: 3/3 exhausted."
    ),
)

_print_result("B", SCENARIO_B)

## Starter exercise

Add a **cost gate**: track the total number of LLM tokens consumed across all repair attempts and halt the loop early if cumulative cost exceeds a budget threshold, emitting a postmortem with `reason='budget_exceeded'`. What schema field would you add and where in the workflow would you check it?

In [ ]:
# ---------------------------------------------------------------------------
# Answer key — cost gate
# ---------------------------------------------------------------------------

# 1. Schema addition
# Add `total_tokens_used` and `budget_exceeded` to HealingResult so callers
# can observe spend and know whether the loop was halted early.

class HealingResultWithBudget(HealingResult):
    """HealingResult extended with token accounting for cost-gated loops."""

    total_tokens_used: int = Field(
        default=0,
        description="Cumulative LLM tokens consumed across all repair attempts.",
    )
    budget_exceeded: bool = Field(
        default=False,
        description="True when the loop was halted early because the token budget was exhausted.",
    )


# 2. Workflow change
# Accumulate resp.usage.total_tokens after every LLM call (classify, propose,
# validate). Before proposing the next repair, check total_tokens >= token_budget.
# If the budget is exceeded, skip straight to _write_postmortem() and return
# HealingResultWithBudget(resolved=False, budget_exceeded=True).

def run_with_budget(
    failure: CIFailure,
    max_retries: int = 3,
    token_budget: int = 5000,
) -> HealingResultWithBudget:
    """
    Bounded self-healing loop with a token-cost gate.

    Halts early and emits a postmortem when cumulative token usage across
    classify + propose + validate calls exceeds `token_budget`.
    """
    client = _get_client()
    total_tokens = 0

    # --- Classify (counts toward budget) ---
    user_content = (
        f"Job: {failure.job_name}\n"
        f"Step: {failure.step_name}\n"
        f"Exit code: {failure.exit_code}\n\n"
        f"Log snippet:\n{failure.log_snippet}"
    )
    resp = client.chat.completions.create(
        model=_MODEL,
        messages=[
            {"role": "system", "content": CLASSIFY_SYSTEM},
            {"role": "user", "content": user_content},
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "FailureClassification",
                "strict": True,
                "schema": FailureClassification.model_json_schema(),
            },
        },
    )
    classification = FailureClassification.model_validate_json(resp.choices[0].message.content)
    total_tokens += resp.usage.total_tokens if resp.usage else 0

    attempts: list[RepairAttempt] = []

    for _ in range(max_retries):
        # --- Budget check BEFORE proposing next repair ---
        if total_tokens >= token_budget:
            postmortem = _write_postmortem(failure, classification, attempts)
            return HealingResultWithBudget(
                job_name=failure.job_name,
                resolved=False,
                attempts_taken=len(attempts),
                final_strategy=None,
                postmortem=postmortem,
                total_tokens_used=total_tokens,
                budget_exceeded=True,
            )

        # Propose
        user_content_repair = json.dumps(
            {
                "failure": failure.model_dump(),
                "classification": classification.model_dump(),
                "prior_attempts": [a.model_dump() for a in attempts],
            },
            indent=2,
        )
        resp_repair = client.chat.completions.create(
            model=_MODEL,
            messages=[
                {"role": "system", "content": REPAIR_SYSTEM},
                {"role": "user", "content": user_content_repair},
            ],
            response_format={
                "type": "json_schema",
                "json_schema": {
                    "name": "RepairAttempt",
                    "strict": True,
                    "schema": RepairAttempt.model_json_schema(),
                },
            },
        )
        attempt = RepairAttempt.model_validate_json(resp_repair.choices[0].message.content)
        attempt.notes = attempt.notes or "pending"
        total_tokens += resp_repair.usage.total_tokens if resp_repair.usage else 0

        # Validate
        user_content_validate = json.dumps(
            {
                "attempt": attempt.model_dump(),
                "original_log_snippet": failure.log_snippet,
            },
            indent=2,
        )
        resp_validate = client.chat.completions.create(
            model=_MODEL,
            messages=[
                {"role": "system", "content": VALIDATE_SYSTEM},
                {"role": "user", "content": user_content_validate},
            ],
            response_format={"type": "json_object"},
        )
        verdict = json.loads(resp_validate.choices[0].message.content)
        attempt.validation_result = verdict.get("validation_result", "inconclusive")
        attempt.notes = verdict.get("notes", attempt.notes)
        total_tokens += resp_validate.usage.total_tokens if resp_validate.usage else 0
        attempts.append(attempt)

        if attempt.validation_result == "pass":
            return HealingResultWithBudget(
                job_name=failure.job_name,
                resolved=True,
                attempts_taken=len(attempts),
                final_strategy=attempt.strategy,
                total_tokens_used=total_tokens,
                budget_exceeded=False,
            )

    # Retries exhausted normally (budget was never hit)
    postmortem = _write_postmortem(failure, classification, attempts)
    return HealingResultWithBudget(
        job_name=failure.job_name,
        resolved=False,
        attempts_taken=len(attempts),
        final_strategy=None,
        postmortem=postmortem,
        total_tokens_used=total_tokens,
        budget_exceeded=False,
    )


# Demonstration (uses a very low budget to trigger early halt on Scenario B)
result_budgeted = run_with_budget(SCENARIO_B, max_retries=3, token_budget=300)
print(f"Resolved:        {result_budgeted.resolved}")
print(f"Budget exceeded: {result_budgeted.budget_exceeded}")
print(f"Tokens used:     {result_budgeted.total_tokens_used}")
print(f"Attempts taken:  {result_budgeted.attempts_taken}")